# Kalpa Health · Brief A · Part 1: Merging

Reads every monthly booking export and writes the two merged files the cleaning notebook starts from:
`old_legacy.csv` (all six cities, legacy system) and `nova.csv` (Chennai and Jaipur, Nova system, one row per test).

Your original code is kept. `# CHANGED` marks a fix, `# ADDED` marks a new line or cell. No cleaning happens here: header rows, Bangalore, rescheduled originals and date filters are left for the cleaning notebook.

# Reading Data and Creating Dataframes city wise

In [ ]:
import os
import pandas as pd
from pathlib import Path
data_dir = Path.cwd()/'datapack'
os.listdir(data_dir)

# ADDED: the merged files are written here (the cleaning notebook reads them from DATA_DIR)
OUT_DIR = Path.cwd()


In [ ]:
for item in data_dir.iterdir():
    print(item.name, "(folder)" if item.is_dir() else "(file)")

##Booking

In [ ]:
booking_path = data_dir /'bookings'

###Legacy

In [ ]:
def read_city_csv_legacy(city_path):
    df_list = []

    # CHANGED: sorted() so every run reads the files in the same order
    for item in sorted(city_path.iterdir()):
        item_path = city_path /item.name
        #print(item_path)
        df = pd.read_csv(item_path, compression="gzip", encoding="latin-1")
        df["source_file"] = item.name        # ADDED: lets us bin by export-file month later (the 4.99% vs 4.97% question)
        df_list.append(df)

    return pd.concat(df_list, ignore_index=True)

####Ahmedabad

In [ ]:
ahmd_path = booking_path / 'legacy' / 'ahmedabad'
df_ahmed = read_city_csv_legacy(ahmd_path)
df_ahmed.head()

####Bengaluru

In [ ]:
beng_path = booking_path / 'legacy' / 'bengaluru'
df_beng = read_city_csv_legacy(beng_path)
df_beng.head()

####Chennai

In [ ]:
chen_path = booking_path / 'legacy' / 'chennai'
df_chen = read_city_csv_legacy(chen_path)
df_chen.head()

####Hyderabad

In [ ]:
hyd_path = booking_path / 'legacy' / 'hyderabad'
df_hyd = read_city_csv_legacy(hyd_path)
df_hyd.head()

####Jaipur

In [ ]:
jai_path = booking_path / 'legacy' / 'jaipur'
df_jai = read_city_csv_legacy(jai_path)
df_jai.head()

####Pune

In [ ]:
pun_path = booking_path / 'legacy' / 'pune'
df_pun = read_city_csv_legacy(pun_path)
df_pun.head()

####Combine the six legacy cities  (ADDED: this step was missing)

In [ ]:
# ADDED
df_legacy = pd.concat([df_ahmed, df_beng, df_chen, df_hyd, df_jai, df_pun], ignore_index=True)
print("Legacy rows:", len(df_legacy))
print(df_legacy.groupby("city").size())     # note: Bengaluru still appears as 'Bangalore' here; the cleaning notebook renames it
df_legacy.to_csv(OUT_DIR / "old_legacy.csv", index=False)
print("Saved old_legacy.csv")

###Nova

Your step-by-step cells for Chennai are kept below as the worked example. The function that follows is the same logic, fixed, and applied to both Nova cities.

In [ ]:
file_path = booking_path /'nova' /'chennai'/'bookings_2026-04.jsonl.gz'

df = pd.read_json(file_path, lines=True, compression="gzip")
print(df.shape)
df.head()

In [ ]:
df_chen.info()

In [ ]:
df.info()

In [ ]:
patient_df = pd.json_normalize(df['patient'])
patient_df.head()

In [ ]:
df_new = pd.concat( [df.drop(columns=['patient']), patient_df], axis=1 )
df_new.head()

In [ ]:
df_new = df_new.explode('tests', ignore_index=True)
df_new.head()

In [ ]:
tests_df = pd.json_normalize(df_new['tests'])
tests_df = tests_df.add_prefix('test_')
tests_df.head()

In [ ]:
df_new = pd.concat([df_new.drop(columns=['tests']), tests_df], axis=1)
df_new.head()

In [ ]:
df_new.info()

In [ ]:
df_new.shape[0]

In [ ]:
df_new.loc[df_new['is_new']]

####Nova reader as one function (CHANGED: the original function used `df_new` inside, which only exists outside it, and never joined the tests back)

In [ ]:
def read_json_nova(city_path):
    df_list = []

    for item in sorted(city_path.iterdir()):
        df = pd.read_json(item, lines=True, compression="gzip")
        patient_df = pd.json_normalize(df['patient'])
        df = pd.concat([df.drop(columns=['patient']), patient_df], axis=1)
        df = df.explode('tests', ignore_index=True)
        # CHANGED: a booking with an empty tests list explodes to NaN. Make it an empty dict so it stays as one
        # row with blank test_code (a visit with zero tests), the same as your Chennai cells above.
        tests_df = pd.json_normalize(df['tests'].apply(lambda t: t if isinstance(t, dict) else {}))
        tests_df = tests_df.add_prefix('test_')
        # CHANGED: tests_df has no 'test_code' column at all if every list in the file is empty
        for col in ["test_code", "test_price"]:
            if col not in tests_df.columns:
                tests_df[col] = pd.NA
        df = pd.concat([df.drop(columns=['tests']), tests_df], axis=1)
        df["source_file"] = item.name      # ADDED
        df_list.append(df)

    return pd.concat(df_list, ignore_index=True)

In [ ]:
# ADDED: both Nova cities, then combine
df_nova_che = read_json_nova(booking_path / 'nova' / 'chennai')
df_nova_jai = read_json_nova(booking_path / 'nova' / 'jaipur')
df_nova = pd.concat([df_nova_che, df_nova_jai], ignore_index=True)

print("Nova rows (one per test line):", len(df_nova))
print("Nova bookings:", df_nova["booking_id"].nunique())
print(df_nova.groupby("city")["booking_id"].nunique())
print("Rows with blank test_code (bookings with no tests):", df_nova["test_code"].isna().sum())
df_nova.to_csv(OUT_DIR / "nova.csv", index=False)
print("Saved nova.csv")

## Checks (ADDED)

In [ ]:
def check(name, ok, detail=""):
    print(("PASS" if ok else "FAIL"), "-", name, detail)

n_legacy_files = sum(1 for _ in (booking_path/'legacy').rglob('*.csv.gz'))
n_nova_files = sum(1 for _ in (booking_path/'nova').rglob('*.jsonl.gz'))
check("legacy files read: 6 cities x 18 months, minus Chennai and Jaipur from April 2026 = 102", n_legacy_files == 102, f"(found {n_legacy_files})")
check("Nova files read: 2 cities x 3 months (Apr-Jun 2026) = 6", n_nova_files == 6, f"(found {n_nova_files})")
check("every legacy row has a booking_id", df_legacy["booking_id"].notna().all())
check("Nova columns include test_code, test_price, status_at_export, rescheduled_from",
      {"test_code","test_price","status_at_export","rescheduled_from"} <= set(df_nova.columns))
check("Nova keeps the original and the replacement of a reschedule (cleaning drops the originals)",
      (df_nova["status_at_export"] == "rescheduled").any())
check("Chennai Nova April: 5,903 bookings and 13,821 rows (your worked example)",
      (df_nova[(df_nova.city=="Chennai") & (df_nova.source_file=="bookings_2026-04.jsonl.gz")]
       .booking_id.nunique() == 5903))